# 数据结构教程：排序与堆——八种内部排序、优先队列与外部排序

排序是数据结构的主干章节，也是考研选择题出题密度最高的地方：给你一串中间状态让你反推算法、让你手算一趟快排、让你画一个堆——考的全是"你有没有真的亲手跑过这些算法"。所以本教程的写法很明确：**每种排序都配一段可以动手验证的手工模拟**，代码全部是能独立编译运行的完整程序。

本讲覆盖三条线：

1. **主线（王道第 8 章）**：插入、交换、选择、归并、基数、计数六大类内部排序；
2. **堆专题（学习路线阶段十三）**：把堆当成独立的优先队列结构来讲透；
3. **第二轮强化（王道 8.7）**：外部排序——败者树、置换-选择排序、最佳归并树。

约定：所有堆相关的下标一律使用 **0-based**（C++ 数组风格），父子公式统一为 `parent(i) = (i-1)/2`、左孩子 `2i+1`、右孩子 `2i+2`，全文不再切换。文中出现的 `49'` 表示第二个值为 49 的元素，用来观察稳定性。

---

## 一、排序的基本概念（8.1）

### 1. 什么是排序

**排序 (Sort)**：重新排列表中的元素，使表中的元素满足按关键字递增或递减有序的过程。

形式化一点：给定 n 个记录 R1..Rn，其关键字分别为 k1..kn，排序就是确定一个排列 p1..pn，使得 kp1 ≤ kp2 ≤ ... ≤ kpn，并按这个排列重排记录。

两个输入要点：

- **关键字 (Key)**：排序依据的字段。可以是整数，也可以是字符串、结构体的某个成员。
- **主关键字 vs 次关键字**：主关键字唯一标识记录（如学号），次关键字可以重复（如成绩）。稳定性问题正是由次关键字重复引出的。

### 2. 稳定性：相等元素的"人身权"

> **定义**：设序列中有两个关键字相等的记录 Ri 和 Rj（i 在 j 之前）。若排序后 Ri 仍在 Rj 之前，则称该排序算法是**稳定的 (Stable)**；否则是不稳定的。

很多初学者觉得"反正都排好序了，相等的谁前谁后有什么关系"。看一个实际场景：

```
报名表已经按提交时间排好：

  张三  98 分   ← 先提交
  李四  98 分   ← 后提交
  王五  95 分

现在要"按分数从高到低"再排一次。

稳定算法的结果：          不稳定算法可能的结果：
  张三  98                李四  98
  李四  98                张三  98
  王五  95                王五  95

张三和李四都是 98，但张三先交卷——稳定算法保住了
"同分先交卷者靠前"这条隐含规则，不稳定算法把它毁了。
```

更工程化的说法：如果排序的关键字只携带部分信息（比如先按姓名排序、再按部门排序，希望部门相同时保持姓名序），那么只有稳定排序能把上一轮的成果保留下来。这就是**多关键字排序可以用多次稳定排序叠加实现**的原因，第八部分的基数排序正是这么干的。

> **核心结论**：稳定性是算法自身的性质，与输入无关。判断一个算法稳不稳，标准做法是构造一个带重复关键字的短序列，手动推演一遍看相等元素会不会互换位置。本讲会为每个不稳定算法给出具体反例。

### 3. 内部排序与外部排序

| 类别 | 数据存放位置 | 关注点 | 典型代表 |
|------|------------|--------|----------|
| **内部排序** | 全程在内存 | 比较和移动的次数（时间复杂度） | 本讲第二～七部分 |
| **外部排序** | 数据在外存（磁盘），内存一次装不下 | 磁盘读写次数（IO 次数） | 本讲第十部分 |

```
内部排序家族分类图：

                    内部排序
      ┌──────────┬──────────┬──────────┐
    插入排序     交换排序     选择排序    其他
    ├─直接插入   ├─冒泡       ├─简单选择   ├─归并排序
    ├─折半插入   └─快速排序   └─堆排序     ├─基数排序
    └─希尔排序                             └─计数排序
```

一个常被忽略的事实：**没有"全能冠军"**。每种排序都有自己的舒适区——快排平均最快但最坏会退化，归并稳定但费内存，堆排省内存但不稳定。第八部分的总表会把这笔账算清楚。

---

## 二、插入排序（8.2）

### 1. 直接插入排序（8.2.1）

#### 1.1 核心思想

想想你整理手牌：摸起一张牌，从右往左扫已经理好的牌，找到它该待的位置插进去。**序列分成"已排序区"和"未排序区"，每趟把未排序区的第一个元素插入已排序区的正确位置。**

```
初始：  [ 49 | 38  65  97  76  13  27  49' ]
         已排序区  ↑ 未排序区
第 i 趟：取出 a[i]，在 a[0..i-1] 中从右往左找位置，
         比 key 大的统统右移一格，然后放入空位。
```

三个要点：

- 第 0 个元素自己构成初始的已排序区，所以趟次从 i = 1 到 n−1，共 **n−1 趟**；
- 找位置时遇到**等于 key 的元素必须停下**（用 `>` 而不是 `>=` 比较）——这是稳定性的命门；
- 移动是"整体右移一格"，不是交换，所以比冒泡省一半以上的赋值。

#### 1.2 手工模拟

对 `{49, 38, 65, 97, 76, 13, 27, 49'}` 逐趟执行（加粗的是本趟插入的元素）：

| 趟次 | 插入元素 | 过程 | 一趟后的结果 |
|------|---------|------|-------------|
| i=1 | 38 | 49>38，49 右移，38 放入队首 | [38, 49, 65, 97, 76, 13, 27, 49'] |
| i=2 | 65 | 49<65，不用动 | [38, 49, 65, 97, 76, 13, 27, 49'] |
| i=3 | 97 | 65<97，不用动 | [38, 49, 65, 97, 76, 13, 27, 49'] |
| i=4 | 76 | 97>76 右移，65<76 停 | [38, 49, 65, **76**, 97, 13, 27, 49'] |
| i=5 | 13 | 97、76、65、49、38 全右移，放到队首 | [13, 38, 49, 65, 76, 97, 27, 49'] |
| i=6 | 27 | 97..38 右移，13<27 停 | [13, 27, 38, 49, 65, 76, 97, 49'] |
| i=7 | 49' | 97、76、65 右移；碰到 49，49 不大于 49，**停下** | [13, 27, 38, 49, **49'**, 65, 76, 97] |

注意最后一步：49' 正好落在 49 的后面，两者的相对次序和初始时一致——这就是"相等就停"带来的稳定性。下面这段程序的实际输出与上表完全一致，你可以改改数字自己玩。

#### 1.3 C++ 实现

核心函数不长，但每一行都有讲究：

In [ ]:
// 直接插入排序：把 a[i] 插入到前面已经有序的 a[0..i-1] 中

In [ ]:
void insertionSort(vector<int>& a) {
    int n = a.size();
    for (int i = 1; i < n; i++) {   // 第 0 个元素自成一个已排序区
        int key = a[i];      // 摸起一张“牌”
        int j = i - 1;
        // 必须用 > 而不是 >=：遇到相等的元素就停下，
        // 这样 key 只会插到相等元素的后面，稳定性才保得住
        while (j >= 0 && a[j] > key) {
            a[j + 1] = a[j]; // 比 key 大的元素整体后移一格
            j--;
        }
        a[j + 1] = key;      // 放进腾出来的空位
    }
}

三处细节值得停一停：趟次从 i=1 开始，因为第 0 个元素天然有序；比较用 `>` 不用 `>=`，相等立刻刹车，这是稳定性的命门；挪动用赋值而非交换，把一段后缀整体平移一格，比冒泡的两两交换省一半以上写入。

验证用的 main 只要几行（加上 `<iostream>`、`<vector>` 两行头文件即是完整程序）：

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    insertionSort(a);
    for (int x : a) cout << x << " ";   // 输出 13 27 38 49 49 65 76 97
    cout << endl;
    return 0;
}

In [ ]:
main();

在循环末尾临时加一句打印数组，得到的每趟状态与上面表格逐行一致。（王道课本用 `a[0]` 当哨兵省去 `j >= 0` 判断，那是顺序表时代的技巧；这里用现代写法，逻辑完全对应。）

#### 1.4 复杂度与稳定性

| 情形 | 发生条件 | 比较次数 | 移动次数 |
|------|---------|---------|---------|
| 最好 | 已经有序 | 每趟只比 1 次，共 n−1 | 0 |
| 最坏 | 完全逆序 | 第 i 趟要比 i 次，共 n(n−1)/2 | 第 i 趟移 i+1 次，共 (n+2)(n−1)/2 |
| 平均 | 随机序列 | 约 n²/4 | 约 n²/4 |

> **核心结论**：直接插入排序时间复杂度最好 O(n)、平均和最坏 O(n²)；空间 O(1)；**稳定**；适用于**基本有序**或**规模小**的序列。它是"基本有序时最快"的典型——越有序，每趟扫得越短。

### 2. 折半插入排序（8.2.2）

做法：既然 a[0..i-1] 已经有序，定位插入点就不用线性扫，改用**折半查找**，比较次数降到约 O(n log₂n)：

In [ ]:
// 折半插入：折半定位 + 整体右移（放进上面同一个程序即可编译）

In [ ]:
void binaryInsertionSort(vector<int>& a) {
    int n = a.size();
    for (int i = 1; i < n; i++) {
        int key = a[i];
        int low = 0, high = i - 1;
        while (low <= high) {                 // 在 a[0..i-1] 里折半查找
            int mid = low + (high - low) / 2;
            if (a[mid] <= key) low = mid + 1; // 相等也往右半边走：保稳定
            else high = mid - 1;
        }
        // 循环结束时 low 就是插入点，把 a[low..i-1] 整体右移一格
        for (int j = i - 1; j >= low; j--) a[j + 1] = a[j];
        a[low] = key;
    }
}

对同一组数据调用它，结果与直接插入完全相同（`13 27 38 49 49 65 76 97`）。

但要认清它的局限：

| 维度 | 直接插入 | 折半插入 |
|------|---------|---------|
| 定位方式 | 从右往左线性扫 | 折半查找 |
| 比较次数 | 依赖初始有序程度，最坏 O(n²) | 约 O(n log₂n)，**与初始状态无关** |
| 移动次数 | O(n²) | **一模一样，还是 O(n²)** |
| 总体量级 | O(n²) | O(n²) |

> **核心结论**：折半插入只优化了"找位置"（比较），没有优化"挪元素"（移动）。元素必须连续存储、整体平移这件事没变，所以总时间量级仍是 O(n²)。它依然是**稳定**的（相等时往右半边插）。顺带一提：折半插入的比较次数与初始状态无关，这是选择题爱挖的坑。

### 3. 希尔排序（8.2.3）

#### 3.1 思想：先让序列"大致有序"

直接插入的两个软肋：逆序时移动步幅太小（每次只挪一格）、大规模乱序时束手无策。1959 年 Donald Shell 的想法是：**先按较大的步长分组做插入排序，再逐步缩小步长，最后一步用步长 1 收尾**。

```
gap = 4 时，下标 {0,4}、{1,5}、{2,6}、{3,7} 各是一组，
组内分别做插入排序（各组交替进行）。

gap 逐渐缩小：4 → 2 → 1

关键洞察：做完 gap=g 的一趟后，序列“按任意间隔 g 看都是有
序的”（称为 g-有序）。前面的功夫不会白费——到 gap=1 那一
趟时，序列已经基本有序，而直接插入恰恰对基本有序序列接近
O(n)。大步长负责长途押送，小步长负责精修。
```

#### 3.2 手工模拟

还是 `{49, 38, 65, 97, 76, 13, 27, 49'}`，n=8，增量取 4、2、1。

**gap = 4**，四组分别是 (0,4)、(1,5)、(2,6)、(3,7)：

```
组 (0,4)：{49, 76}  →  有序，不动
组 (1,5)：{38, 13}  →  交换 → 13 在前
组 (2,6)：{65, 27}  →  交换 → 27 在前
组 (3,7)：{97, 49'} →  交换 → 49' 在前

结果：[49, 13, 27, 49', 76, 38, 65, 97]
```

**gap = 2**，偶数组 {0,2,4,6}={49,27,76,65}，奇数组 {1,3,5,7}={13,49',38,97}：

```
偶数组插入排序：49,27,76,65 → 27,49,65,76
奇数组插入排序：13,49',38,97 → 13,38,49',97

结果：[27, 13, 49, 38, 65, 49', 76, 97]
```

**gap = 1**，就是普通直接插入排序，但此时序列已基本有序，只挪了几下：

```
i=3 的 38 越过 49；i=5 的 49' 越过 65，停在 49 之后。
最终：[13, 27, 38, 49, 49', 65, 76, 97]
```

#### 3.3 C++ 实现

外层管增量的收缩，内层就是换了个步长的直接插入：

In [ ]:
void shellSort(vector<int>& a) {
    int n = a.size();
    // 增量从 n/2 开始每轮减半，最后一轮 gap=1 退化为直接插入
    for (int gap = n / 2; gap > 0; gap /= 2) {
        // 从下标 gap 开始往后扫：每个元素与自己所在组的
        // 前面元素做插入。各组在这里是“交替”进行的，效果
        // 与“一组排完再排下一组”完全一致
        for (int i = gap; i < n; i++) {
            int key = a[i];
            int j = i - gap;             // 同组的上一个元素
            while (j >= 0 && a[j] > key) {
                a[j + gap] = a[j];       // 组内大步后移
                j -= gap;
            }
            a[j + gap] = key;
        }
    }
}

两处边界别写歪：组内回退是 `j -= gap`，不是 `j--`；gap 序列必须以 1 收尾，否则序列可能没排完就停了。

配上几行的 main（需 `<iostream>`、`<vector>`）：

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    shellSort(a);
    for (int x : a) cout << x << " ";   // 13 27 38 49 49 65 76 97
    cout << endl;
    return 0;
}

In [ ]:
main();

在 gap 循环末尾临时加一句打印数组，跑出来的三趟状态与 3.2 的手推逐字一致：`gap=4 → [49, 13, 27, 49', 76, 38, 65, 97]`，`gap=2 → [27, 13, 49, 38, 65, 49', 76, 97]`，`gap=1 → [13, 27, 38, 49, 49', 65, 76, 97]`。

#### 3.4 复杂度与稳定性

- **时间复杂度依赖增量序列**，这是希尔排序最特殊的地方。数学分析至今没有完全解决，经验结论：n 在几千到几万规模时约为 **O(n^1.3)**；Hibbard 增量（1, 3, 7, ..., 2^k−1）可证最坏 O(n^1.5)；Shell 最原始的对半增量最坏会退化到 O(n²)。
- 空间 O(1)。
- **不稳定**。相等元素被分进不同组，各自的迁移互不知情，跨组之后相对次序就可能颠倒。一个最小反例：`{3, 2a, 2b, 1}`，gap=2 时 2b 进偶数组、2a 进奇数组，gap=2 趟后变成 `[2b, 1, 3, 2a]`；gap=1 时 2a 往前挪，碰到相等的 2b 就停——最终 `[1, 2b, 2a, 3]`，2a、2b 的先后关系被颠倒了。

---

## 三、交换排序（8.3）

交换排序的共同特征：**通过交换逆序偶来消除逆序**。冒泡只交换相邻元素，快排的交换可以跳很远——这一字之差，复杂度天壤之别。

### 1. 冒泡排序（8.3.1）

#### 1.1 实现与提前终止

每趟从前往后依次比较相邻元素，逆序就交换；一趟下来**当前最大的元素像气泡一样沉到（冒到）末尾**，下一趟就不用再看它了。

In [ ]:
void bubbleSort(vector<int>& a) {
    int n = a.size();
    for (int i = 0; i < n - 1; i++) {          // 最多 n-1 趟
        bool swapped = false;                  // 本趟是否发生过交换
        for (int j = 0; j < n - 1 - i; j++) {  // 尾部 i 个已就位，不再参与
            if (a[j] > a[j + 1]) {             // 相等不交换 => 稳定
                swap(a[j], a[j + 1]);
                swapped = true;
            }
        }
        if (!swapped) break;                   // 整趟无交换 => 已有序，提前收工
    }
}

三层防线各司其职：外层 `n-1` 趟只是上限；内层 `n-1-i` 让已沉底的大家伙不再陪跑；`swapped` 负责"发现已有序就提前下班"。相等不交换，稳定性保住。

In [ ]:
// 需要 <iostream> <vector>

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    bubbleSort(a);
    for (int x : a) cout << x << " ";   // 13 27 38 49 49 65 76 97
    cout << endl;

    vector<int> b = {1, 2, 3, 4, 5};
    bubbleSort(b);   // 已有序：第一趟零交换，立即终止（最好情况）
    return 0;
}

In [ ]:
main();

#### 1.2 手工模拟（前两趟）

对 `{49, 38, 65, 97, 76, 13, 27, 49'}`：

```
第 1 趟：49,38 交换 → 49,65 走 → 65,97 走 → 97,76 交换
        → 97,13 交换 → 97,27 交换 → 97,49' 交换
  结果：[38, 49, 65, 76, 13, 27, 49', | 97]     97 归位

第 2 趟：38,49 走 → 49,65 走 → 65,76 走 → 76,13 交换
        → 76,27 交换 → 76,49' 交换
  结果：[38, 49, 65, 13, 27, 49', | 76, 97]     76 归位
```

后续各趟依次把 65、49、38 送到位；第 5 趟全程没有发生交换，`swapped` 保持 false，算法提前终止。最终 `[13, 27, 38, 49, 49', 65, 76, 97]`。

> **核心结论**：冒泡最好 O(n)（一趟发现有序就停，比较 n−1 次）、平均最坏 O(n²)；空间 O(1)；**稳定**。提前终止优化使"趟数与初始状态有关"——这本身就是考点：冒泡的排序趟数取决于输入序列。

### 2. 快速排序（8.3.2）

#### 2.1 总体思想

> **每趟选定一个枢轴 (pivot)，通过一趟划分把序列切成两部分：左边全 ≤ 枢轴，右边全 ≥ 枢轴，枢轴本身落到它的最终位置上；然后对左右两部分递归做同样的事。**

它是分治法在排序里的代表作。难点全在一趟划分 (partition) 上：怎么在不借助额外数组的前提下完成切分？教材标准答案是**挖坑法**。

#### 2.2 挖坑法 partition：双指针怎么走

把 `a[low]`（枢轴）想象成被"挖走"了，`low` 处留下一个坑。规则三句话：

1. **右指针先动**：high 从右往左扫，找第一个比枢轴**小**的元素，扔进左边的坑——坑转移到 high 处；
2. **左指针后动**：low 从左往右扫，找第一个比枢轴**大**的元素，扔进右边的坑——坑又转移回 low 处；
3. 两指针相遇时，坑不再转移，把枢轴放进去。相遇点就是枢轴的最终位置。

```
为什么右指针先走？
因为第一个坑开在最左边（枢轴是从 low 挖走的），
必须先从右边找一个“小的”来填左坑。
反过来，若枢轴取自区间末尾，则应左指针先动。
```

#### 2.3 手工模拟一轮 partition

对 `{49, 38, 65, 97, 76, 13, 27, 49'}` 做第一趟划分，pivot = 49，low=0，high=7，坑在 0：

| 步骤 | 动作 | 数组状态（□ 表示坑） |
|------|------|---------------------|
| 初始 | 取出 pivot=49 | [□, 38, 65, 97, 76, 13, 27, 49'] |
| ① | high 左扫：49'≥49 跳过；27<49 → 填坑，坑移到 6 | [27, 38, 65, 97, 76, 13, □, 49'] |
| ② | low 右扫：27、38 均 ≤49；65>49 → 填坑，坑移到 2 | [27, 38, □, 97, 76, 13, 65, 49'] |
| ③ | high 左扫：13<49 → 填坑，坑移到 5 | [27, 38, 13, 97, 76, □, 65, 49'] |
| ④ | low 右扫：97>49 → 填坑，坑移到 3 | [27, 38, 13, □, 76, 97, 65, 49'] |
| ⑤ | high 左扫：76≥49 跳过；low==high==3，相遇 | [27, 38, 13, □, 76, 97, 65, 49'] |
| 结束 | 枢轴 49 入坑 | **[27, 38, 13, 49, 76, 97, 65, 49']** |

一趟结束：49 站在了下标 3，也就是它在最终有序序列里的位置。**快排的一趟至少确定一个元素的最终位置**——这是考试判断"中间状态属于哪种算法"的重要指纹。

#### 2.4 完整实现

先把最难啃的 partition 单独写出来：

In [ ]:
// 挖坑法：对 a[low..high] 划分，枢轴取 a[low]，返回其最终下标

In [ ]:
int partition(vector<int>& a, int low, int high) {
    int pivot = a[low];   // 枢轴被“挖走”，low 处留下第一个坑
    while (low < high) {
        // 1) high 从右往左扫，找第一个比枢轴小的，填进左边的坑
        while (low < high && a[high] >= pivot) high--;
        a[low] = a[high];
        // 2) low 从左往右扫，找第一个比枢轴大的，填进右边的坑
        while (low < high && a[low] <= pivot) low++;
        a[high] = a[low];
    }
    a[low] = pivot;       // 相遇处就是枢轴的最终位置
    return low;
}

两个内层 while 都挂着 `low < high` 的保护，防止指针互相越界；`>=` 与 `<=` 让等于枢轴的元素安坐原位，不来回折腾。有了这块基石，递归骨架反而平淡：

In [ ]:
void quickSort(vector<int>& a, int low, int high) {
    if (low < high) {                        // 至少两个元素才需要排
        int pos = partition(a, low, high);   // 一趟划分：枢轴归位
        quickSort(a, low, pos - 1);          // 递归处理左半
        quickSort(a, pos + 1, high);         // 递归处理右半
    }
}

// 需要 <iostream> <vector>

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    quickSort(a, 0, (int)a.size() - 1);
    for (int x : a) cout << x << " ";   // 13 27 38 49 49 65 76 97
    cout << endl;
    return 0;
}

In [ ]:
main();

对本例，四次划分依次得到：`[27,38,13,49,76,97,65,49']` → `[13,27,38,...]` → 右半 `[49',65,76,97]` → `[49',65]` 微调，最终有序。注意最终结果里 49 排在 49' 前面纯属巧合（它们恰好相邻且没被迫交换过）——**快排整体上是不稳定的**，相等元素的相对次序没有任何保障。

除了挖坑法，另一路经典写法是 Hoare 的**交换法**：左右指针各自越过不合格元素后直接 swap（而不是填坑），最后再把枢轴换到相遇点。两者趟完的状态可以不同（枢轴落点可能不一样），但对"划分正确"这件事是等价的。考试若指定"以第一个元素为枢轴进行一趟快排"，按挖坑法手算最稳妥。

#### 2.5 性能分析

| 指标 | 结果 | 条件说明 |
|------|------|---------|
| 最好时间 | O(n log₂n) | 每趟划分都近似对半分，递归树是平衡的，共 log n 层，每层 O(n) |
| 最坏时间 | O(n²) | 每趟划分都"一头沉"，比如**序列已经有序/逆序**却总取首元素当枢轴——n 趟划分每趟只减少一个元素 |
| 平均时间 | O(n log₂n) | 随机序列下期望划分基本均衡 |
| 空间 | 平均 O(log₂n)，最坏 O(n) | 递归栈深度 = 递归树高度 |
| 稳定性 | **不稳定** | 远距离交换会打乱相等元素的相对次序 |

> **核心结论**："快速排序是所有内部排序中**平均性能最优**的"——这句话成立的前提是：待排序列足够随机、每次划分大致均衡。它的常数因子在 O(n log n) 级别的排序中最小。一旦前提被破坏（基本有序还固定取端点当枢轴），立刻退化成 O(n²)。工程上的补救：**三者取中**（首、中、尾取中位数当枢轴）或**随机选取枢轴**，以及小区间切换成插入排序。

---

## 四、选择排序（8.4）

选择排序的哲学：**每一趟从未处理区间里挑出该放这里的元素，直接放过去**。"挑选"的方式决定了它是简单选择（线性扫着挑）还是堆排序（用堆挑）。

### 1. 简单选择排序（8.4.1）

#### 1.1 实现

第 i 趟在 a[i..n-1] 中找最小值，与 a[i] 交换。

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

In [ ]:
void selectionSort(vector<int>& a) {
    int n = a.size();
    for (int i = 0; i < n - 1; i++) {
        int minIdx = i;
        for (int j = i + 1; j < n; j++)   // 只记录下标，不急着换
            if (a[j] < a[minIdx]) minIdx = j;
        if (minIdx != i)
            swap(a[i], a[minIdx]);        // 一趟最多一次交换
    }
}

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    selectionSort(a);
    for (int x : a) cout << x << " ";
    cout << endl;   // 输出 13 27 38 49 49 65 76 97
    return 0;
}

In [ ]:
main();

#### 1.2 特点与稳定性

- **比较次数恒为 n(n−1)/2，与初始状态无关**——哪怕输入已经完全有序，内层循环照样从头扫到尾。这是它与插入、冒泡最大的性格差异（那两位在有序输入时都能提前收工）。
- 移动次数很少：最好 0 次，最坏 3(n−1) 次（每次 swap 三次赋值）。适合"比较便宜、移动昂贵"的场景（比如元素是巨大的结构体）。

**不稳定，反例 `{2, 2', 1}`：**

```
第 1 趟：全表最小是 1（下标 2），与 a[0]=2 交换
        [1, 2', 2]
2' 跑到了 2 的前面——它们的相对次序被一次远距离交换破坏了。
```

### 2. 堆排序（8.4.2_1）

简单选择每趟都要线性扫一遍来找最小值，太笨。堆排序的升级思路：**用大根堆这种数据结构把"找最值"从 O(n) 加速到 O(1)**（堆顶就是最值，取走后重调只需 O(log n)）。堆的完整性质、图解和推导统一放在第九部分，这里先给结论性的三句话：

1. **大根堆**：一颗完全二叉树存在数组里，任何父结点 ≥ 其孩子，于是 a[0] 就是全局最大值；
2. **排序流程** = 建堆（O(n)）+ 反复"堆顶与末尾交换、堆规模缩一、下滤重调"（共 n−1 轮，每轮 O(log n)）；
3. 每轮都有一个元素（当前最大值）落到最终位置——所以堆排和快排、简单选择一样，**一趟归位至少一个元素**。

#### 2.1 完整代码

心脏是**下滤**——前提"某结点的两棵子树已经是堆"，把它沉到合法位置：

In [ ]:
// 下滤（大根堆）：把 a[i] 沉到它在堆中的合法位置。
// 前提：i 的左右子树各自已经是大根堆

In [ ]:
void siftDown(vector<int>& a, int i, int size) {
    while (true) {
        int l = 2 * i + 1, r = 2 * i + 2;  // 左右孩子下标（0-based）
        int largest = i;                   // 在 i 和它的孩子中挑最大
        if (l < size && a[l] > a[largest]) largest = l;
        if (r < size && a[r] > a[largest]) largest = r;
        if (largest == i) break;           // i 已不小于两个孩子：到位
        swap(a[i], a[largest]);            // 与更大的孩子交换，继续下沉
        i = largest;
    }
}

最容易写错的是孩子那两行：必须**先把两个孩子比出大的，再决定跟谁交换**——只跟左孩子比较就和右孩子换，是新手最常见的 bug；`l < size / r < size` 负责剔除不存在的那一侧。

有了下滤，其余部分短得惊人：

In [ ]:
// 自底向上建堆：从最后一个非叶结点倒着逐个下滤

In [ ]:
void buildMaxHeap(vector<int>& a) {
    for (int i = (int)a.size() / 2 - 1; i >= 0; i--)
        siftDown(a, i, a.size());
}

In [ ]:
void heapSort(vector<int>& a) {
    buildMaxHeap(a);
    for (int end = (int)a.size() - 1; end > 0; end--) {
        swap(a[0], a[end]);      // 堆顶(当前最大值)与堆尾交换
        siftDown(a, 0, end);     // 对剩下 end 个元素重调堆顶
    }
}

// 需要 <iostream> <vector>

In [ ]:
int main() {
    vector<int> a = {53, 17, 78, 9, 45, 65, 87, 32};
    heapSort(a);
    for (int x : a) cout << x << " ";   // 输出 9 17 32 45 53 65 78 87
    cout << endl;
    return 0;
}

In [ ]:
main();

建堆起点 `n/2 - 1` 是最后一个非叶结点，再往右全是叶子、无需调整；排序循环把 `end` 当作堆规模传给下滤，已就位的尾部自然被排除在堆外。下面模拟的每一步，都可以拿这个程序加打印来核对。

#### 2.2 手工模拟

初始数组 `{53, 17, 78, 9, 45, 65, 87, 32}`（n=8），对应的完全二叉树：

```
              53(0)
            /      \
         17(1)     78(2)
         /   \     /   \
      09(3) 45(4) 65(5) 87(6)
      /
    32(7)

最后一个非叶结点：下标 n/2-1 = 3（值 09）
```

**建堆阶段**：按下标 3 → 2 → 1 → 0 依次下滤。

```
siftDown(3)：09 只有左孩子 32，32>09，交换
  → [53, 17, 78, 32, 45, 65, 87, 09]

siftDown(2)：78 的孩子 65、87，大的 87>78，交换
  → [53, 17, 87, 32, 45, 65, 78, 09]

siftDown(1)：17 的孩子 32、45，大的 45>17，交换
  → [53, 45, 87, 32, 17, 65, 78, 09]

siftDown(0)：53 的孩子 45、87 → 87 上来；53 落到 2 号位，
             它的新孩子 65、78 → 78 上来；53 落到 6 号位（叶子）
  → [87, 45, 78, 32, 17, 65, 53, 09]   ← 大根堆建成

              87(0)
            /      \
         45(1)     78(2)
         /   \     /   \
      32(3) 17(4) 65(5) 53(6)
      /
    09(7)
```

**排序阶段**：反复"堆顶 ↔ 末尾交换，规模减一，下滤堆顶"。每一轮结束后的状态如下（竖线右侧为已就位部分）：

| 轮次 | 交换走掉的堆顶 | 一轮后的数组 |
|------|--------------|-------------|
| 1 | 87 | [78, 45, 65, 32, 17, 9, 53 \| 87] |
| 2 | 78 | [65, 45, 53, 32, 17, 9 \| 78, 87] |
| 3 | 65 | [53, 45, 9, 32, 17 \| 65, 78, 87] |
| 4 | 53 | [45, 32, 9, 17 \| 53, 65, 78, 87] |
| 5 | 45 | [32, 17, 9 \| 45, 53, 65, 78, 87] |
| 6 | 32 | [17, 9 \| 32, 45, 53, 65, 78, 87] |
| 7 | 17 | [9 \| 17, 32, 45, 53, 65, 78, 87] |

最终升序 `[9, 17, 32, 45, 53, 65, 78, 87]`。以上每一步都与 2.1 程序的实际输出一致。

> **核心结论**：堆排序建堆 O(n)、排序 n−1 轮每轮 O(log n)，总计 **O(n log₂n)，且最坏情况也是如此**；空间 O(1)；**不稳定**（远距离交换）；不适合链表（下滤需要随机访问孩子下标）。它是"最坏情况下仍保证 O(n log n) 且额外空间 O(1)"的唯一主流选择。

### 3. 堆的插入与删除（8.4.2_2）

堆作为动态结构，还要支持插入（**上滤**）和删除堆顶（**下滤**）。这两个操作是优先队列的灵魂，实现、图解与边界讨论统一放在第九部分第 3 节，此处记住方向即可：

- **插入用上滤**：新元素先挂在堆尾，然后"比爹大就和爹换"，一层层浮上去；
- **删除堆顶用下滤**：堆顶弹出后，把堆尾元素搬到堆顶补位，然后往下沉。

---

## 五、归并排序（8.5.1）

### 1. 核心思想：分治 + 合并

> 把序列一分为二，两半各自排好序，再把两个有序表**合并 (merge)** 成一个有序表。递归下去，直到每边只剩一个元素（天然有序）。

整个算法的地基是"合并两个有序表"：双指针分别指向两队队首，每次比较取较小者出队——像拉链一样合拢。

### 2. 手工模拟（按"趟"来看）

对 `{49, 38, 65, 97, 76, 13, 27, 49'}`，从"相邻两两归并"的视角看，一共 3 趟（⌈log₂8⌉ = 3）：

```
初始：  [49] [38] [65] [97] [76] [13] [27] [49']
第 1 趟：[38 49] [65 97] [13 76] [27 49']      两两归并
第 2 趟：[38 49 65 97] [13 27 49' 76]          四四归并
第 3 趟：[13 27 38 49 49' 65 76 97]            合并为整体
```

盯住最后一趟里 49 和 49' 的处理：合并 `[38 49 65 97]` 与 `[13 27 49' 76]` 时，49 与 49' 相遇，取 `<=` 让**左边队伍的 49 先出**——相等时左队优先，稳定性就是这么保住的。归并是"结构上就稳定"的算法，不像快排那样靠小心翼翼的比较符。

### 3. C++ 实现

地基是合并函数——双指针像拉链一样把两条有序队伍合拢：

In [ ]:
// 合并 a[lo..mid] 与 a[mid+1..hi] 这两个有序段（借助辅助数组 tmp）

In [ ]:
void mergeTwo(vector<int>& a, int lo, int mid, int hi, vector<int>& tmp) {
    int i = lo, j = mid + 1, k = lo;
    while (i <= mid && j <= hi)
        // 取 <=：两边相等时优先取左段，稳定性由此保证
        tmp[k++] = (a[i] <= a[j]) ? a[i++] : a[j++];
    while (i <= mid) tmp[k++] = a[i++];   // 左段有剩余，整段搬过去
    while (j <= hi)  tmp[k++] = a[j++];   // 右段有剩余
    for (k = lo; k <= hi; k++) a[k] = tmp[k];  // 写回原数组
}

两个收尾 while 不能省：一支队伍打空后，另一支剩下的本来就有序，整段搬走即可。分治骨架反而平平无奇：

In [ ]:
void mergeSort(vector<int>& a, int lo, int hi, vector<int>& tmp) {
    if (lo >= hi) return;                 // 区间长度 <= 1：天然有序
    int mid = lo + (hi - lo) / 2;         // 写成 lo+(hi-lo)/2 防溢出
    mergeSort(a, lo, mid, tmp);           // 先排左半
    mergeSort(a, mid + 1, hi, tmp);       // 再排右半
    mergeTwo(a, lo, mid, hi, tmp);        // 最后合并
}

// 需要 <iostream> <vector>

In [ ]:
int main() {
    vector<int> a = {49, 38, 65, 97, 76, 13, 27, 49};
    vector<int> tmp(a.size());
    mergeSort(a, 0, (int)a.size() - 1, tmp);
    for (int x : a) cout << x << " ";   // 输出 13 27 38 49 49 65 76 97
    cout << endl;
    return 0;
}

In [ ]:
main();

**非递归（自底向上）版本**的思路一句话就能说清：不做递归分解，直接从"每个元素自成一段"开始，按段宽 1、2、4、8... 循环调用 `mergeTwo`，正好对应上面手工模拟的三趟。两种写法趟数、结果完全相同，非递归版省掉递归栈。

### 4. 性能与归并树

| 指标 | 结论 |
|------|------|
| 时间 | 每趟合并总量 O(n)，共 ⌈log₂n⌉ 趟 ⇒ **O(n log₂n)，最好最坏都一样** |
| 空间 | 辅助数组 O(n) + 递归栈 O(log n) ⇒ **O(n)**，这是它最大的短板 |
| 稳定性 | **稳定**（相等取左） |
| 链表适配 | **最适合链表的 O(n log n) 排序**：合并改成改指针，连 O(n) 辅助数组都不需要 |

**归并树与比较次数**：合并两个长度为 m、n 的有序表，最少比较 min(m,n) 次（一支被打空），最多 m+n−1 次。整个归并过程画成一棵二叉树（叶子是初始元素，内部结点是合并操作），树的形态决定了总比较量——"把短的段尽早合并、长的段晚点合"能省比较次数。把这个想法推到 k 路归并的外部排序场景，就是第十部分的最佳归并树。

---

## 六、基数排序（8.5.2）

### 1. 多关键字排序：MSD 与 LSD

前面的排序都在"比较"关键字，基数排序压根不比较——它按关键字的**每一位**（或每一个子关键字）做"分配-收集"。

多关键字排序有两个方向：

| 策略 | 全称 | 做法 | 特点 |
|------|------|------|------|
| **MSD** | 最高位优先 (Most Significant Digit) | 先按最高位分组，组内递归再按次高位分 | 递归分桶，实现复杂 |
| **LSD** | 最低位优先 (Least Significant Digit) | 从最低位开始，整趟"分配-收集"，逐位往高位推进 | 一遍遍线性扫，实现简单，**教材标准做法** |

LSD 能成立的前提：**除最后一趟外，每一趟的收集都必须稳定**。道理很直观——第 1 趟按个位排好的次序，要在第 2 趟按十位收集时依然保留（十位相同的那些数之间，个位的功劳才不至于白费）。所以桶要用**队列**（先进先出），不能用栈。

### 2. 手工模拟（LSD）

对 `{53, 27, 36, 15, 69, 42}`（d = 2 位，基数 r = 10）：

**第 1 趟：按个位分配**

```
桶0: 空        桶5: 15
桶2: 42        桶6: 36
桶3: 53        桶7: 27
               桶9: 69
收集（0→9 依次出队）：42, 53, 15, 36, 27, 69
```

**第 2 趟：按十位分配**

```
桶1: 15        桶4: 42
桶2: 27        桶5: 53
桶3: 36        桶6: 69
收集：15, 27, 36, 42, 53, 69   ← 有序
```

注意两点：第一趟结束后整体**并不有序**（42 排在了 15 前面），必须凑齐 d 趟才算完；十位相同的数之间（本例没有，但比如 51、55）靠的正是第 1 趟保留下来的个位次序。

### 3. C++ 实现

In [ ]:
// LSD 基数排序（要求 a 中元素为非负整数）：从最低位开始，逐位“分配-收集”

In [ ]:
void radixSortLSD(vector<int>& a) {
    if (a.empty()) return;
    int maxv = *max_element(a.begin(), a.end());
    // exp 依次取 1(个位)、10(十位)、100(百位)... 共 d 趟
    for (long long exp = 1; maxv / exp > 0; exp *= 10) {
        vector<queue<int>> buckets(10);        // r=10 个队列桶
        for (int x : a)
            buckets[(x / exp) % 10].push(x);   // 分配：按当前位入队
        int idx = 0;
        for (int d = 0; d < 10; d++)           // 收集：0 号桶到 9 号桶
            while (!buckets[d].empty()) {      // 队列出队 => 保稳定
                a[idx++] = buckets[d].front();
                buckets[d].pop();
            }
    }
}

两个设计决定值得咀嚼：桶必须是**队列**，先进先出才能让第 1 趟排好的个位次序在第 2 趟幸存；循环条件 `maxv / exp > 0` 自动把最大值的位数当成了趟数 d。

In [ ]:
// 需要 <iostream> <vector> <queue> <algorithm>

In [ ]:
int main() {
    vector<int> a = {53, 27, 36, 15, 69, 42};
    radixSortLSD(a);
    for (int x : a) cout << x << " ";   // 输出 15 27 36 42 53 69
    cout << endl;
    return 0;
}

In [ ]:
main();

第一趟（按个位）收集完是 `42, 53, 15, 36, 27, 69`——整体并不有序；第二趟（按十位）后才得到最终结果，过程与第 2 节的手推一致。

### 4. 性能

设关键字有 d 位，基数为 r（十进制 r=10），共 n 个元素：

> **核心结论**：基数排序共 d 趟“分配+收集”，每趟 O(n+r)，总计 **O(d(n+r))**；空间 **O(n+r)**（桶中要暂存 n 个元素，另有 r 个队列）；**稳定**；不进行比较，也不受 O(n log n) 下界约束。适用条件：**n 很大、d 很小**（位数少的非负整数、定长字符串）。d 大了（比如长字符串）就不划算了。

---

## 七、计数排序（8.5.3）

### 1. 适用前提

> 序列的关键字是**范围很小的非负整数**（值域 [0, k)，k 不太大）。这时根本不用比较——数一数每个值出现几次，直接按次数重写就行。

### 2. C++ 实现（稳定版）

朴素做法（统计个数后按序重写关键字）丢掉了同值元素之间的先后信息和附带数据。要做稳定版，用**前缀和 + 从后往前放置**：

In [ ]:
// 稳定版计数排序：要求关键字是 [0, k) 内的非负整数

In [ ]:
void countingSort(vector<int>& a) {
    if (a.empty()) return;
    int n = a.size();
    int k = *max_element(a.begin(), a.end()) + 1;   // 值域大小
    vector<int> cnt(k, 0);
    for (int x : a) cnt[x]++;                       // 1) 计数
    for (int i = 1; i < k; i++)
        cnt[i] += cnt[i - 1];                       // 2) 前缀和 => 名次信息
    vector<int> b(n);
    // 3) 从后往前放：cnt[v] 此时指向“<=v 的最后一个位置”，
    //    同值元素中后扫描的反而落在更靠后的格子里，
    //    原始相对次序被原样保留 —— 这就是稳定性
    for (int i = n - 1; i >= 0; i--)
        b[--cnt[a[i]]] = a[i];
    a = b;
}

三步走：计数、求前缀和、逆序回放。第三步为什么从后往前？正着放的话，同值元素会被按"扫描顺序"倒着塞回去，稳定性恰好被毁掉——这是本算法最精巧也最爱考的一处。

In [ ]:
// 需要 <iostream> <vector> <algorithm>

In [ ]:
int main() {
    vector<int> a = {4, 2, 4, 1, 0, 2, 3};
    countingSort(a);
    for (int x : a) cout << x << " ";   // 输出 0 1 2 2 3 4 4
    cout << endl;
    return 0;
}

In [ ]:
main();

### 3. 性能

> **核心结论**：本节稳定版计数排序时间 **O(n+k)**、空间 **O(n+k)**（输出数组需要 O(n)；若只按次数重写关键字可降为 O(k)，但不保稳定）。k 一大（比如给 32 位整数排序）空间直接爆炸，所以它只服务"值域小"的场景。它还有一个重要兼职：**基数排序对每一位的"分配-收集"本质上就可以用一次稳定的计数排序来实现**。计数排序也是稳定的（稳定版写法）。

---

## 八、排序总表与选型指南

### 1. 总表（背下来）

| 算法 | 最好 | 平均 | 最坏 | 空间 | 稳定 | 适合链表 | 适用规模/场景 |
|------|------|------|------|------|------|---------|--------------|
| 直接插入 | O(n) | O(n²) | O(n²) | O(1) | 是 | **适合**（免移动，改指针即可） | 小规模、基本有序 |
| 折半插入 | O(n log n)¹ | O(n²) | O(n²) | O(1) | 是 | 一般 | 小规模；比较贵、移动便宜时 |
| 希尔 | ~O(n^1.3) | 依赖增量 | O(n²)² | O(1) | 否 | 否（需随机访问） | 中等规模的通用备选 |
| 冒泡 | O(n) | O(n²) | O(n²) | O(1) | 是 | 可以 | 教学；几乎有序的小数据 |
| 快速排序 | O(n log n) | O(n log n) | **O(n²)** | O(log n)³ | 否 | 不便（需反向扫描） | 大规模随机数据的首选 |
| 简单选择 | O(n²) | O(n²) | O(n²) | O(1) | 否 | 可以 | 比较廉价、移动昂贵的元素 |
| 堆排序 | O(n log n) | O(n log n) | **O(n log n)** | **O(1)** | 否 | 否（需下标访问） | 大规模；只要部分最值；内存苛刻 |
| 归并排序 | O(n log n) | O(n log n) | O(n log n) | **O(n)** | **是** | **最佳**（合并改指针） | 要求稳定 + O(n log n)；外存的基石 |
| 基数排序 | O(d(n+r)) | 同左 | 同左 | O(n+r) | 是 | 可以（桶用队列） | n 大、d 小的多关键字/整数 |
| 计数排序（稳定版） | O(n+k) | 同左 | 同左 | O(n+k) | 是 | 无关紧要 | 值域很小的整数 |

¹ 折半插入的**比较**次数最好也是 O(n log n) 量级且与初始无关；移动仍是 O(n²)。
² 取决于增量序列，Shell 原始增量的最坏情形。
³ 递归栈深：平均 O(log n)，最坏 O(n)。

### 2. 什么场景选什么

拿真实需求过一遍这张表：

- **n 很小**（几十以内）：别纠结，直接插入或简单选择，常数小、代码短，快排在这种规模上反而吃亏；
- **基本有序**：直接插入或冒泡——它们能感知有序性，逼近 O(n)；千万别用固定取首元素的快排，这是退化重灾区；
- **大规模随机数据、追求速度**：快速排序，平均性能之王；担心恶意输入就用随机枢轴；
- **要求稳定，又要 O(n log n)**：归并排序，没有第二选项（堆排、快排都不稳定）；
- **内存紧张、还要保证最坏 O(n log n)**：堆排序，原地且抗退化；实时系统也偏爱它——性能上界锁得死；
- **海量整数/定长字符串，位数不多**：基数排序，绕开比较排序的下界；
- **非负小整数**：计数排序，线性时间；
- **只要前 K 大/小、流式数据、频繁取最值**：根本不要完整排序，上堆（第九、Top-K 实战）；
- **数据在磁盘上装不进内存**：归并思想的天下（第十部分）。

### 3. 考研高频题型：看中间状态反推算法

题目会给"某算法跑到一半的序列"，让你判断是哪种排序。按下面的指纹库排查：

| 观察到的特征 | 指向 |
|-------------|------|
| 一趟后有**至少一个元素到达最终位置**（通常是枢轴/最值站中间或两端） | 快排、简单选择、堆排、冒泡 |
| 前缀始终局部有序（前 i+1 个有序，但里面的元素未必是全局最小的几个） | 插入类（直接/折半） |
| 相隔某个 gap 的子序列各自有序 | 希尔 |
| 成双成对的有序块，块长按 2、4、8 翻倍 | 归并 |
| 一趟内全是**相邻交换**、最大值沉底 | 冒泡 |
| 一趟内只发生**一次**远距离交换（最小值换到队首） | 简单选择 |

两条铁律：

1. **插入排序的任何时刻，都不能保证任何一个元素处于最终位置**（除非恰好撞上）。看到"前 3 个有序但 13 这种小值还在后面"，别选插入；
2. 区分"一趟"的定义：快排一趟 = 一次完整 partition；冒泡一趟 = 一轮完整的相邻扫描；希尔一趟 = 一个固定 gap 下的全部组。

---

## 九、堆专题（学习路线阶段十三）

第四部分为了讲堆排序，已经把 siftDown 和建堆的代码亮过了。这个专题换个身份看堆：**它首先是一个优先队列数据结构，排序只是顺手的事**。学完这节，你要能独立回答：堆为什么快？怎么插入删除？STL 的 priority_queue 有哪些坑？

### 1. 堆的定义与数组表示

> **堆 (Heap)**：一颗**完全二叉树**，且满足堆序性质——
> **大根堆**：任意结点 ≥ 其孩子（根是最大值）；
> **小根堆**：任意结点 ≤ 其孩子（根是最小值）。

两个关键词缺一不可。"完全二叉树"保证了它可以**无损地塞进数组**：按层序编号存放，父子关系用纯算术就能算出来。本文统一 0-based 下标（C++ 数组天然如此；若用 1-based 则孩子是 2i 和 2i+1，只是平移一下，本质相同）：

```
数组:  [87, 45, 78, 32, 17, 65, 53, 09]        n = 8

              87(0)
            /      \                parent(i) = (i-1)/2
         45(1)     78(2)            left(i)   = 2i+1
         /   \     /   \            right(i)  = 2i+2
      32(3) 17(4) 65(5) 53(6)
      /
    09(7)

叶子结点：下标 [n/2, n-1]，即 4..7（17、65、53、09）
最后一个非叶结点：n/2 - 1 = 3
```

> **核心结论**：完全二叉树 ⇒ 数组存储零浪费；堆序性质 ⇒ a[0] 永远是极值。这两条合起来，就是堆的一切效率的来源：找最值 O(1)，插入删除 O(log n)——树高只有 ⌊log₂n⌋。

### 2. 下滤 siftDown：堆的心脏

**下滤 (sift-down / 向下调整)**：某个结点比它的孩子小（违反了大根堆性质），让它一路和"更大的那个孩子"交换，沉到合法位置。前提是它的两棵子树**已经是堆**——这个前提决定了自底向上建堆的正确性。

以建堆时 siftDown(0) 为例（数组 [53, 45, 87, 32, 17, 65, 78, 9]），看 53 的下沉路线：

```
第 1 步：53 的孩子是 45、87 → 挑大的 87 与 53 交换

              [53]                      53 悬空
            /      \
          45       [87]
         /  \      /   \
       32   17   65    78
       /
      9

第 2 步：53 的新位置(2)的孩子是 65、78 → 挑大的 78 交换

          87
        /      \
      45       [53]   ← 孩子变 65、78
     /  \      /  \
   32   17   65   78
   /
  9

第 3 步：53 落在下标 6，已是叶子，结束
最终：[87, 45, 78, 32, 17, 65, 53, 9]
```

代码就是第四部分堆排序程序里的 `siftDown`，要点回顾：**两个孩子先比出大的，再决定跟谁换**；某一侧孩子不存在就跳过；`largest == i` 即停。每下沉一层做常数次比较，最多沉 ⌊log₂n⌋ 层 ⇒ **O(log n)**。

### 3. 上滤 insert 与删除堆顶（8.4.2_2）

**插入（上滤）**：新元素先挂到堆尾（保持完全二叉树形状），然后一路"比爹大就和爹换"地浮上去。

```
在上面的堆里插入 90：

第一步：挂到堆尾（下标 8）          第二步起：一路向上

          87                              [90]←起点8，爹是3号(32)
        /      \                         90>32，换！爹变1号(45)
      45        78                       90>45，换！爹变0号(87)
     /  \      /   \                     90>87，换！到根，结束
   32    17  65    53                  最终 90 登顶
   /
 [90](8)
```

**删除堆顶**：堆顶（极值）弹出去之后，**不能直接拿孩子顶上来**（会让完全二叉树破洞）。正确做法：把**堆尾最后一个元素**搬到根补位（形状完好），再对根下滤。

先写上滤。新元素挂在堆尾后，唯一要做的就是"比爹大就和爹换"：

In [ ]:
// 上滤插入：新元素从堆尾往上浮（配合第四部分 2.1 节的 siftDown 使用）

In [ ]:
void heapInsert(vector<int>& heap, int x) {
    heap.push_back(x);              // 先挂堆尾，保持完全二叉树形状
    int i = heap.size() - 1;
    while (i > 0) {
        int parent = (i - 1) / 2;
        if (heap[parent] >= heap[i]) break;   // 爹不小：到位
        swap(heap[parent], heap[i]);          // 比爹大：换上去继续浮
        i = parent;
    }
}

注意父结点是 `(i-1)/2`——0-based 的专属公式，从 1-based 教材里顺手抄成 `i/2` 是高频 bug。删除堆顶则反过来：

In [ ]:
// 删除并返回堆顶（最大值），需要第四部分的 siftDown 配合

In [ ]:
int extractMax(vector<int>& heap) {
    int top = heap[0];
    heap[0] = heap.back();   // 堆尾补到根：形状完好，性质交给下滤修复
    heap.pop_back();         // 移除原堆尾
    if (!heap.empty())       // 还有剩余才需要调整（防空堆越界）
        siftDown(heap, 0, heap.size());
    return top;
}

把这两个函数与 4.2 节的 siftDown 放进同一个文件就能跑：

In [ ]:
// 需要 <iostream> <vector>

In [ ]:
int main() {
    vector<int> heap = {87, 45, 78, 32, 17, 65, 53, 9};  // 已是大根堆
    heapInsert(heap, 90);
    for (int x : heap) cout << x << " ";   // 输出 90 87 78 45 17 65 53 9 32
    cout << endl;
    cout << extractMax(heap) << endl;      // 输出 90
    for (int x : heap) cout << x << " ";   // 输出 87 45 78 32 17 65 53 9（复原）
    cout << endl;
    return 0;
}

In [ ]:
main();

> **易错点**：删除堆顶时"拿左孩子顶上去"是错的（树会残缺）；上滤时忘记更新 `i = parent`、或者把 `(i-1)/2` 写成 `i/2`（那是 1-based 公式混进来了），都是高频 bug。

### 4. 自底向上建堆为什么是 O(n)

倒序对所有非叶结点下滤，直觉上"每个结点都可能沉 log n 层"，很多人以为建堆是 O(n log n)。错在哪？**绝大多数结点根本沉不了几层**。

按"离叶子的最大距离"（高度 h）分层统计，n 个结点的完全二叉树里：

| 高度 h（叶子为 0） | 该层结点数占比 | 单点下滤最多走 h 层 | 该层总代价 |
|------------------|--------------|--------------------|-----------|
| 0（叶子） | 约 1/2 | 0（根本不下滤） | 0 |
| 1 | 约 1/4 | ≤ 1 | n/4 · 1 |
| 2 | 约 1/8 | ≤ 2 | n/8 · 2 |
| 3 | 约 1/16 | ≤ 3 | n/16 · 3 |
| ... | ... | ... | ... |

总代价 ≤ Σ_{h≥0} (n/2^(h+1)) · h = (n/2) · Σ_{h≥0} h/2^h。由 Σ h/2^h = 2（经典级数），得上界 **n**，即 **O(n)**。

一句话记忆：**一半的结点是叶子，零成本；越往上层人越少，虽然每个人走得远，但"人数 ÷ 2^h"衰减得比"h"增长得快**。相比之下，逐个插入建堆（n 次上滤）是 O(n log n)——大量元素要浮很远。所以建堆永远用自底向下。

### 5. std::priority_queue：用法与陷阱

STL 已经替你封装好了堆，头文件 `<queue>`。基本用法先跑通：

In [ ]:
#include <iostream>
#include <queue>
#include <vector>
#include <functional>   // greater 需要
using namespace std;

In [ ]:
int main() {
    priority_queue<int> maxPQ;                            // 默认是大根堆
    maxPQ.push(3); maxPQ.push(1); maxPQ.push(4);
    cout << maxPQ.top() << endl;                          // 4

    priority_queue<int, vector<int>, greater<int>> minPQ; // 小根堆写法
    minPQ.push(3); minPQ.push(1); minPQ.push(4);
    cout << minPQ.top() << endl;                          // 1
    return 0;
}

In [ ]:
main();

装自定义类型时需要提供一个比较器。语义务必看清——`operator()` 返回 true 表示 a 的优先级**低于** b，a 会被排到更靠近堆底的位置：

In [ ]:
struct Task {
    string name;
    int priority;
};

In [ ]:
struct TaskCmp {
    bool operator()(const Task& a, const Task& b) const {
        return a.priority < b.priority;   // priority 大的先出队
    }
};

In [ ]:
int main() {                              // 另需 <string> <queue> <vector>
    priority_queue<Task, vector<Task>, TaskCmp> tasks;
    tasks.push({"写作业", 2});
    tasks.push({"救火", 10});
    tasks.push({"摸鱼", 1});
    cout << tasks.top().name << endl;     // 救火
    return 0;
}

In [ ]:
main();

五个必知陷阱：

1. **默认是大根堆**。`priority_queue<int>` 等价于指定了 `less<int>`，top() 是最大值。想要小根堆必须写全三个模板参数并用 `greater<int>`；
2. **比较器语义与 sort 相反的直觉**。sort 的 cmp 返回 true 表示"a 应排在前面"；priority_queue 的 cmp 返回 true 表示"a 排在 b 后面（更低优先级）"。所以"按 priority 从小到大排"的结构体，放进 priority_queue 要写 `a.priority < b.priority` 才能让大的先出队——写反了堆顶就不是你想要的那个；
3. **不支持 decrease-key**。Dijkstra 这类算法需要"降低堆内某元素的距离值"，priority_queue 做不到。惯用替代是**懒惰删除**：允许同一结点带着新值重复入队，出队时比对是否过期，过期就丢弃；
4. **接口极简**：只有 push / pop / top / empty / size。`top()` 只读不删，`pop()` 只删不返回——取堆顶要 `top()` 和 `pop()` 配合；
5. **底层容器默认 vector**，push/pop 均摊 O(log n)。迭代遍历出来的顺序不是有序的，别指望。

### 6. Top-K 实战：最大的 K 个数用小根堆

问题：n 个数里找最大的 K 个（n 很大，K 很小，比如从十亿条日志里找最慢的 100 条请求）。全排序要 O(n log n)，堆的做法只要 **O(n log K)** 时间、**O(K)** 空间。

> **思路**：维护一个**容量为 K 的小根堆**当"晋级名单"。堆顶是名单里**最弱的一个**，相当于守门员。每个新元素来挑战：比守门员还弱，直接淘汰；比守门员强，踢走守门员、新元素入场。扫完全部数据，堆里剩下的就是最大的 K 个。

核心逻辑不到二十行：

In [ ]:
// 返回 data 中最大的 k 个元素（降序）。
// minHeap 是容量不超过 k 的小根堆：堆顶是名单里最弱的“守门员”

In [ ]:
vector<int> topK(const vector<int>& data, int k) {
    if (k <= 0) return {};
    priority_queue<int, vector<int>, greater<int>> minHeap;
    for (int x : data) {
        if ((int)minHeap.size() < k) {
            minHeap.push(x);            // 名单没满：先来先进
        } else if (x > minHeap.top()) {
            minHeap.pop();              // 比守门员强：踢掉守门员
            minHeap.push(x);
        }
    }
    vector<int> res;
    while (!minHeap.empty()) {          // 出来是升序
        res.push_back(minHeap.top());
        minHeap.pop();
    }
    reverse(res.begin(), res.end());    // 转成降序
    return res;
}

调用端只有几行正经代码：

In [ ]:
// 需要 <iostream> <vector> <queue> <algorithm> <functional>

In [ ]:
int main() {
    vector<int> data = {3, 8, 1, 9, 5, 2, 7, 6, 4, 0};
    for (int x : topK(data, 3)) cout << x << " ";   // 输出 9 8 7
    cout << endl;
    return 0;
}

In [ ]:
main();

> **常见误区**：求最大的 K 个，凭直觉想用大根堆——错。大根堆的堆顶是名单里**最强**的，新元素比它弱时你没法判断该不该放进来（它可能比名单里其他 K−1 个都强），根本没有高效的淘汰判据。**求最大的 K 个 ⇒ 小根堆；求最小的 K 个 ⇒ 大根堆**，方向永远和直觉拧着。这也是面试和考研都爱的送分/送命题。

---

## 十、外部排序（8.7，第二轮强化内容）

> 这一节是第二轮才需要吃透的内容。第一轮只需要记住一句：数据在磁盘上装不进内存时，排序的主角不再是"比较和移动"，而是"读写磁盘"。

### 1. 为什么磁盘 IO 决定一切（8.7.1 – 8.7.2）

假设 900 个记录装不进内存，内存一次只能容 3 个。做法只能是：

```
① 分批读入内存 → 内部排序（比如逐段用置换-选择生成长归并段）
② 把每段有序数据写回磁盘 —— 得到若干“初始归并段”(merge segment)
③ 反复“从磁盘读几段 → 内存中多路归并 → 写回磁盘”
   直到只剩一个有序段
```

耗时构成：**内部排序/归并的计算时间 + 外存信息读写时间 + 若干趟归并的内部时间**。磁盘读写比内存慢几个数量级，所以总时间的支配项是 IO，而 **IO 总量正比于归并趟数 × 文件大小**。

于是外部排序的全部优化都指向一个目标——**减少归并趟数**：

- **减少初始段个数 r**（段越长段越少）→ 置换-选择排序（第 3 节）；
- **每趟归并更多路数 k**（r 个段做 k 路平衡归并只需 ⌈log_k r⌉ 趟）→ 多路归并（第 2 节）；
- **安排段的归并顺序，让短段少跑几趟** → 最佳归并树（第 4 节）。

例：r=9 个初始段，2 路归并要 ⌈log₂9⌉=4 趟，3 路只要 2 趟。但增大 k 有代价：k 路归并需要在内存里放 k 个输入缓冲区 + 1 个输出缓冲区，k 太大则缓冲区被切得太碎；更要命的是，每输出一个记录都要从 k 个段首中选最小者，朴素扫描是 O(k)——这正是败者树的用武之地。

### 2. 多路平衡归并与败者树（8.7.3）

#### 2.1 问题

k 路归并过程中，内存里同时放着 k 个段的"当前队首"，每输出一个记录就要选出其中最小者。朴素做法扫一遍 k 个候选：O(k) 每次；整个归并输出 n 个记录就是 O(nk)。k 越大（趟数越少的初衷）这里反而越慢——**败者树把每次选择降到 O(log₂k)**，鱼和熊掌兼得。

#### 2.2 败者树的结构

> **败者树 (Tree of Loser)**：一棵完全二叉树。叶子是 k 个段的当前记录；每个内部结点记的是**两个孩子中"败者"（较大者）所在的段号**，胜者继续向上比；树根之上再设一个结点 ls[0]，记录最终冠军（最小值所在段号）。k 个叶子的完全二叉树恰有 k−1 个内部结点，编号 ls[1..k-1]。

与"锦标赛"对照着理解：冠军一路过关斩将，但树上记的全是他打败过的对手。好处是：下次比赛只有新冠军所在的那条路径需要重赛——因为其他人都输给过"旧阵容"，实力没变。

#### 2.3 ASCII 图解：一棵 5 路败者树

5 个归并段当前的队首记录：段0=2，段1=4，段2=3，段3=5，段4=11（各段后续记录暂记为段0:{2,**6**,9...}，段2:{3,**8**,...}）。

k=5 的叶子挂在 9 个结点的完全二叉树上（内部 ls[1..4]，叶子 b[0..4]）：

```
初建过程（自底向上比，共 4 次比较 = k-1 次）：

  ls[4]：b3=5 vs b4=11 → 胜者段3，记败者 ls[4]=4
  ls[3]：b1=4 vs b2=3  → 胜者段2，记败者 ls[3]=1
  ls[2]：ls[4]的胜者(段3,值5) vs b0=2 → 胜者段0，记败者 ls[2]=3
  ls[1]：段0(值2) vs ls[3]的胜者(段2,值3) → 胜者段0，记败者 ls[1]=2
  冠军段0 写入 ls[0]=0

                 ┌────────┐
                 │ ls[1]=2│  败者：段2(值3)
                 └───┬────┘
              ┌──────┴──────┐
         ┌────┴────┐   ┌────┴────┐
         │ ls[2]=3 │   │ ls[3]=1 │   败者：段3(值5) / 段1(值4)
         └────┬────┘   └────┬────┘
           ┌──┴──┐        ┌─┴─┐
      ┌────┴──┐  │        │   │
      │ls[4]=4│ b0=2      b1=4  b2=3   败者：段4(值11)
      └───┬───┘
        ┌─┴───┐
       b3=5  b4=11

  ls[0] = 0   ← 冠军：段0 的 2（全局最小）
```

**输出冠军并调整**：输出 2 之后，段0 的下一个记录 6 进入 b0 的位置。只需沿 b0 → ls[2] → ls[1] 这条路径向上重赛：

```
  与 ls[2] 记的旧败者段3(值5) 比：6>5，段0 落败 → ls[2]=0，段3 晋级
  段3(值5) 到根，与 ls[1] 记的旧败者段2(值3) 比：5>3，段3 落败
     → ls[1]=3，冠军是段2 → ls[0]=2

                 ┌────────┐
                 │ ls[1]=3│
                 └───┬────┘
              ┌──────┴──────┐
         ┌────┴────┐   ┌────┴────┐
         │ ls[2]=0 │   │ ls[3]=1 │
         └────┬────┘   └────┬────┘
           ┌──┴──┐        ┌─┴─┐
      ┌────┴──┐  │        │   │
      │ls[4]=4│ b0=6      b1=4  b2=3
      └───┬───┘
        ┌─┴───┐
       b3=5  b4=11

  ls[0] = 2   ← 新冠军：段2 的 3
```

验证：当前五个队首 {6,4,3,5,11}，最小确实是段2 的 3。整个过程**只做了 2 次比较**（路径高度），与 k 无关地封顶在 ⌈log₂k⌉——k=100 路时每次选择最多比 7 次，而不是扫 99 次。

> **核心结论**：败者树内部结点记"败者的段号"、ls[0] 记冠军。初建 O(k)；此后每输出一个记录，调整只走一条叶到根的路径，比较次数 ⌈log₂k⌉。k 路归并的总选择代价从 O(nk) 降到 O(n log k)，这才让"加大归并路数、减少趟数"真正划算。

### 3. 置换-选择排序（8.7.4）

#### 3.1 思想

内部排序法生成初始段时，段长最多等于内存容量——太短了。置换-选择排序的聪明之处：**每输出一个记录，就立刻从输入文件补一个进来**（"置换"），让工作区始终满员。新补的记录若不小于刚输出的记录，就还有资格加入当前段——段因此越长越长，直到工作区里所有记录都比刚输出的记录小，才被迫另起一段。

规则整理（工作区容量 w）：

1. 读入 w 个记录填满工作区；
2. 反复：输出工作区中**不小于上一输出**的最小记录；同时读入下一个记录——若它 ≥ 刚输出的记录，正常参战；否则打上标记"冻结"，留给下一段；
3. 当工作区内全是冻结记录时，当前段结束；解冻，开始新段；
4. 输入耗尽后，把工作区剩余记录按同样规则清空。

#### 3.2 手工模拟（内存容量 w = 3）

输入：`27, 8, 34, 18, 63, 14, 44, 6, 23, 82, 10, 50`。撇号表示冻结。表中工作区一栏均为**该步结束后**的状态（已拿走本次输出、已放入本次读入）。

| 步骤 | 工作区 WA | 本次输出 | 读入 | 说明 |
|------|-----------|---------|------|------|
| 初载 | 27, 8, 34 | — | — | 填满 |
| 1 | 27, 34, 18 | **8** | 18 | 输出最小者 8 |
| 2 | 27, 34, 63 | **18** | 63 | 18≥8，续段 |
| 3 | 34, 63, 14' | **27** | 14 | 14<27，冻结 |
| 4 | 63, 14', 44 | **34** | 44 | 冻结的不参赛；44≥34 |
| 5 | 63, 14', 6' | **44** | 6 | 6<44，冻结 |
| 6 | 14', 6', 23' | **63** | 23 | 23<63，冻结 → 全冻结！ |
| 7 | 14, 23, 82 | 段1 结束，解冻后输出 **6** | 82 | 全部解冻，新段开始 |
| 8 | 23, 82, 10' | **14** | 10 | 10<14，冻结 |
| 9 | 82, 10', 50 | **23** | 50 | |
| 10 | 82, 10' | **50** | 输入完 | 非冻结的只剩 82 |
| 11 | 10' | **82** | — | 输出后全冻结，段2 结束 |
| 12 | 空 | 段3：**10** | — | 残余冻结记录自成一段 |

得到三个初始归并段：

```
段1：8, 18, 27, 34, 44, 63   （长度 6）
段2：6, 14, 23, 50, 82       （长度 5）
段3：10                      （长度 1）
```

内存只装得下 3 个记录，却产出了长 6、长 5 的段。

> **核心结论**：置换-选择排序生成的初始归并段**平均长度是工作区容量的 2 倍**（2w）。初始段更长 ⇒ 段数 r 更少 ⇒ 归并趟数 ⌈log_k r⌉ 更小。它不改变归并阶段的任何东西，纯粹把"起点"垫高了。

### 4. 最佳归并树（8.7.5）

有了 r 个长短不一的初始段、定了归并路数 k，**先归并哪几段**也有讲究：归并树越像哈夫曼树（短的段离根越近、参与合并的次数越少），总的记录读写量越小。构造方法就是第 13 讲哈夫曼树的 k 叉推广，此处不赘述，只补外部排序特有的两件事。

**其一，虚段。** 哈夫曼树要求每次合并凑满 k 个分支；若段数不合适，硬凑会让个别段多跑冤枉路。设初始段有 u 个：

> **补充规则**：若 `(u−1) mod (k−1) == 0`，不需要虚段；否则补充 `d = (k−1) − [(u−1) mod (k−1)]` 个长度为 0 的**虚段**，一起参与哈夫曼构造。

**其二，看一个完整例子。** 设 k=3（三路归并），四个段长度为 {9, 6, 2, 3}。u=4：(4−1) mod 2 = 1 ≠ 0，需补 d = 2 − 1 = **1 个虚段**（长度 0）。

```
不补虚段（错误示范）：只能 2+3+6=11，再 9+11=20
  第一次归并没有凑满 3 路，9 号段被“晾”在外面多跑了一趟

              (20)
             /    \
           (9)   (11)
                 /|\
               (2)(3)(6)

  各段参与归并的读取量 WPL = 9×1 + (2+3+6)×2 = 31

补 1 个虚段 0（正确做法）：
              (20)
             / | \
           (5)(6)(9)
          /|\
        (0)(2)(3)

  WPL = (0+2+3)×2 + (6+9)×1 = 25  <  31
```

虚段的本质：让"第一次归并恰好凑满 k 路"，从而使整棵归并树成为严格的 k 叉哈夫曼树，总 IO 最省。

### 5. 外部排序总流程小结

```
磁盘大文件
    │  分批读入（工作区 w）
    ▼
置换-选择排序 ──► r 个较长的初始归并段（平均长 2w）
    │
    │  按 (u-1)%(k-1) 决定是否补虚段
    ▼
最佳归并树（k 叉哈夫曼）──► 定下归并顺序，总 IO 最小
    │
    │  共 ⌈log_k r⌉ 趟
    ▼
多路平衡归并（k 路输入缓冲 + 1 路输出缓冲）
    │  每次选最小：败者树，O(log k)
    ▼
磁盘上的唯一一个有序文件
```

三件武器各司其职：**置换-选择**管"起点"（段数少），**最佳归并树**管"顺序"（短段少跑），**败者树**管"速度"（每趟归并内部的选择开销）。

---

## 十一、考试重点速查 + 练习题

### 1. 考前 60 秒速查

- 稳定的：直接插入、折半插入、冒泡、归并、基数、计数。**不稳定的：希尔、快排、简单选择、堆排**（口诀：*"希快选堆不安稳"*）。
- 最坏仍是 O(n log n)：归并、堆排。平均最快：快排（前提：划分均衡）。
- 空间 O(1)：插入类、冒泡、选择、希尔、堆排。O(n)：归并。O(log n)：快排（递归栈）。
- 比较次数与初始状态无关：简单选择（恒 n(n−1)/2）、折半插入的比较、基数（不比较）。
- 一趟至少归位一个元素：快排、简单选择、堆排、冒泡。插入与归并做不到。
- 越有序越快的：直接插入（O(n)）、冒泡（提前终止）。
- 建堆 O(n)，堆排序 O(n log n) 且最坏如此；删除堆顶用"堆尾补位 + 下滤"；插入用上滤。
- 堆是完全二叉树存进数组；0-based 孩子 2i+1、2i+2，父 (i−1)/2。
- 外部排序瓶颈是 IO；趟数 ⌈log_k r⌉；败者树每次调整 ⌈log₂k⌉ 次比较；置换-选择平均段长 2w；(u−1) mod (k−1)≠0 时补 (k−1)−[(u−1) mod (k−1)] 个虚段。

### 2. 练习题

**【题 1】中间状态反推。** 初始序列 `{36, 25, 48, 12, 65, 43, 20, 58}`，经某算法处理后，第一趟变为 `{20, 25, 48, 12, 65, 43, 36, 58}`，第二趟变为 `{20, 12, 48, 25, 65, 43, 36, 58}`。这是哪种排序？

> **解析**：对比初始与第一趟：只有位置 0 和 6 的元素（36 与 20）互换了，其余纹丝不动——一趟只发生**一次**交换，且换过来的是全表最小值 20。这是简单选择排序的独家指纹。逐一排除：冒泡一趟后最大值 65 应沉底且全程相邻交换（实际会是 `{25,36,12,48,43,20,58,65}`）；插入一趟只会让前两个元素局部有序（`{25,36,48,...}`）；快排一趟后枢轴 36 应站在最终位置（挖坑法结果是 `{20,25,12,36,65,43,48,58}`）。第二趟把剩余部分的最小值 12 换到位置 1，再次吻合"每趟选一个最值放到前端"的行为。**答案：简单选择排序。**

**【题 2】手算一趟快排。** 对 `{49, 38, 65, 97, 76, 13, 27, 49'}` 以第一个元素为枢轴做一趟（一次）划分，写出结果。

> **解析**：按挖坑法（见 2.3 节的完整表格）：49 挖走后，27 从右边填进坑，65 填去右边，13 填回来，97 填去右边，随后高低指针在 3 号位相遇。结果：**`{27, 38, 13, 49, 76, 97, 65, 49'}`**，枢轴 49 位于下标 3，即它的最终位置。检查：左边 {27,38,13} 全 ≤ 49，右边 {76,97,65,49'} 全 ≥ 49。若题目问"再对左右子区间各做一趟"，左半以 27 为枢轴得 `{13,27,38}`，右半以 76 为枢轴得 `{49',65,76,97}`。

**【题 3】建堆画图。** 把 `{53, 17, 78, 9, 45, 65, 87, 32}` 调整成大根堆，写出数组；随后输出堆顶并重新调整，再写出数组。

> **解析**：n=8，从最后一个非叶结点（下标 3）倒序下滤：3 号 09↔32；2 号 78↔87；1 号 17↔45；0 号 53 依次与 87、78 交换落到叶子。**建堆结果：`[87, 45, 78, 32, 17, 65, 53, 9]`**
>
> ```
>             87
>           /    \
>         45      78
>        /  \    /  \
>      32   17  65   53
>     /
>    9
> ```
> 输出 87：堆尾 9 补到根，下滤——9 与孩子 45、78 中大的 78 交换，再与 65 交换，停在 5 号位。**结果：`[78, 45, 65, 32, 17, 9, 53]`**（堆规模变为 7）。

**【题 4】败者树调整。** 5 路归并，各段当前队首为段0=2、段1=4、段2=3、段3=5、段4=11。(1) 写出初建的败者树（各 ls 值）；(2) 输出冠军后段0 的下一个记录是 6，写出调整后的 ls 与新冠军，共比较几次？

> **解析**：(1) 自底向上：ls[4]=4（5 胜 11），ls[3]=1（3 胜 4），ls[2]=3（2 胜 5），ls[1]=2（2 胜 3），冠军 **ls[0]=0**。(2) 6 进入 b0 后只沿 b0→ls[2]→ls[1] 重赛：6>5 故 ls[2]=0；5>3 故 ls[1]=3，新冠军 **ls[0]=2**（段2 的 3）。ls[3]、ls[4]、b1、b2、b3、b4 全部不动。**共 2 次比较**（≤⌈log₂5⌉=3），而非全树重建。

**【题 5】性质辨析（多选）。** 下列排序算法中，最坏情况下时间复杂度仍为 O(n log n) 且**稳定**的有：A. 快速排序 B. 堆排序 C. 归并排序 D. 希尔排序

> **解析**：最坏 O(n log n) 的是 B 和 C（A 最坏 O(n²)，D 依赖增量、某些增量下更差），但堆排不稳定。同时满足两个条件的只有 **C. 归并排序**——代价是 O(n) 辅助空间。

**【题 6】构造不稳定性反例。** 请举出一个长度为 3 的序列，说明简单选择排序不稳定。

> **解析**：取 `{2, 2', 1}`（2 与 2' 关键字相等）。第 1 趟全表扫描找最小值，1 在下标 2，与下标 0 的 2 交换，得 `{1, 2', 2}`。这一次远距离交换让 2' 越到了 2 前面，相等元素的原始次序被破坏，故不稳定。反例的要领：**相等元素之一必须恰好处在"会被最小值换出去"的位置上**。

---

> **最后的话**：排序这一章没有难在"想不出"的地方，难在"差一点"。差一点的 partition、差一点的孩子选择、差一位的下标公式，都会让程序悄悄出错。把本讲每个手工模拟自己在纸上推一遍，再用配套代码核对——考研卷子上让你默写的，从来不是代码，而是这些一步一步走过来的中间状态。祝学习顺利！